In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
  !git clone -b feature/phase-6-app https://github.com/SanduniHerath/diabetic-retinopathy-cv-assignment.git
  %cd diabetic-retinopathy-cv-assignment

Cloning into 'diabetic-retinopathy-cv-assignment'...
remote: Enumerating objects: 366, done.
remote: Counting objects: 100% (176/176), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 366 (delta 70), reused 129 (delta 40), pack-reused 190 (from 1)
Receiving objects: 100% (366/366), 72.31 MiB | 42.61 MiB/s, done.
Resolving deltas: 100% (123/123), done.
/kaggle/working/diabetic-retinopathy-cv-assignment


In [3]:
  !pip install torch torchvision numpy matplotlib opencv-python scikit-learn datasets

In [5]:
  !mkdir -p data/raw/train_images
  !cp /kaggle/input/competitions/aptos2019-blindness-detection/train.csv data/raw/
  !cp /kaggle/input/competitions/aptos2019-blindness-detection/train.csv data/raw/train_images/
  !cp -r /kaggle/input/competitions/aptos2019-blindness-detection/train_images/* data/raw/train_images/

In [6]:
!python src/organize_dataset.py


[1] Loading CSV from: /kaggle/working/diabetic-retinopathy-cv-assignment/data/raw/train_images/train.csv
     Loaded 3662 records | Columns: ['id_code', 'diagnosis']

[2] Creating class directories under: /kaggle/working/diabetic-retinopathy-cv-assignment/data/organized
     No_DR                -> /kaggle/working/diabetic-retinopathy-cv-assignment/data/organized/No_DR
     Mild                 -> /kaggle/working/diabetic-retinopathy-cv-assignment/data/organized/Mild
     Moderate             -> /kaggle/working/diabetic-retinopathy-cv-assignment/data/organized/Moderate
     Severe               -> /kaggle/working/diabetic-retinopathy-cv-assignment/data/organized/Severe
     Proliferative_DR     -> /kaggle/working/diabetic-retinopathy-cv-assignment/data/organized/Proliferative_DR

[3] Organising 3662 images into class folders ...
  Organised 500/3662 images ...
  Organised 1000/3662 images ...
  Organised 1500/3662 images ...
  Organised 2000/3662 images ...
  Organised 2500/3662 image

In [13]:
!python src/split_dataset.py


PHASE 1: STRATIFIED TRAIN / VAL / TEST DATASET SPLIT
  Input Directory   : data/organized
  Output Directory  : data/split
  Ratios (T/V/Te)   : 0.70 / 0.15 / 0.15
  Random Seed       : 42
  Dry Run           : False
--------------------------------------------------------------------------------
[INFO] Discovered 3662 total images across 5 classes.

           APTOS 2019 DATASET STRATIFIED TRAIN / VAL / TEST SPLIT SUMMARY
Label  | Class Name        | Total   | Train (70%)    | Val (15%)      | Test (15%)    
-------------------------------------------------------------------------------------
0      | No_DR             | 1805    |  1264 ( 70.0%) |   271 ( 15.0%) |   270 ( 15.0%)
1      | Mild              | 370     |   259 ( 70.0%) |    56 ( 15.1%) |    55 ( 14.9%)
2      | Moderate          | 999     |   699 ( 70.0%) |   150 ( 15.0%) |   150 ( 15.0%)
3      | Severe            | 193     |   135 ( 69.9%) |    29 ( 15.0%) |    29 ( 15.0%)
4      | Proliferative_DR  | 295     |   206 ( 

In [14]:
!rm -rf data/organized

In [15]:
!python src/add_external_data.py --max-per-class 500


  EYEPACS SUPPLEMENTARY DATA INGESTION (PHASE 5 EXTENSION)

[1] Verifying dataset split integrity under: /kaggle/working/diabetic-retinopathy-cv-assignment/data/split
  [GUARD] Validation (/kaggle/working/diabetic-retinopathy-cv-assignment/data/split/val) and Test (/kaggle/working/diabetic-retinopathy-cv-assignment/data/split/test) splits are strictly locked.
     Target Directory : /kaggle/working/diabetic-retinopathy-cv-assignment/data/split/train
     Isolation Policy : Val and Test directories are NEVER modified.

[2] Initial training counts baseline:
     Class 0 (No_DR           ):  1264 images
     Class 1 (Mild            ):   259 images
     Class 2 (Moderate        ):   699 images
     Class 3 (Severe          ):   135 images
     Class 4 (Proliferative_DR):   206 images

[3] Ingesting EyePACS supplementary samples ...

[Engine] Initializing Hugging Face dataset: Lelihu/eyepacs
         Target Labels : [1, 3, 4] -> ['Mild', 'Severe', 'Proliferative_DR']
         Quota Limit 

In [17]:
!df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  9.4G   11G  48% /kaggle/working


In [18]:
!python src/augmentation.py


  FILESYSTEM VERIFICATION  (live scan of /kaggle/working/diabetic-retinopathy-cv-assignment/data/split/train )
  Equivalent to:  ls <class_dir>/ | wc -l  for each class
  Class                 Files on disk
  -------------------- --------------
  No_DR                         1,264
  Mild                            759
  Moderate                        699
  Severe                          635
  Proliferative_DR                706
  -------------------- --------------
  TOTAL                         4,063


  DYNAMIC MULTIPLIERS  (target = 1,500 images per class)
  Class                Live count   Multiplier    Projected   vs target
  -------------------- ----------   ----------   ----------   ---------
  No_DR                     1,264            1x        1,264       -236
  Mild                        759            2x        1,518   +      18
  Moderate                    699            2x        1,398       -102
  Severe                      635            2x        1,270       -

In [21]:
!rm -rf reports/augmentation_examples
!df -h /kaggle/working

Filesystem      Size  Used Avail Use% Mounted on
/dev/loop2       20G  9.4G   11G  48% /kaggle/working


In [22]:
!git pull

remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 912 bytes | 456.00 KiB/s, done.
From https://github.com/SanduniHerath/diabetic-retinopathy-cv-assignment
   e2c3779..c44d0ae  feature/phase-6-app -> origin/feature/phase-6-app
Updating e2c3779..c44d0ae
Fast-forward
 src/augmentation.py | 21 +++++++++++++++++++++
 1 file changed, 21 insertions(+)


In [24]:
!python src/augmentation.py


  FILESYSTEM VERIFICATION  (live scan of /kaggle/working/diabetic-retinopathy-cv-assignment/data/split/train )
  Equivalent to:  ls <class_dir>/ | wc -l  for each class
  Class                 Files on disk
  -------------------- --------------
  No_DR                         1,264
  Mild                            759
  Moderate                        699
  Severe                          635
  Proliferative_DR                706
  -------------------- --------------
  TOTAL                         4,063


  DYNAMIC MULTIPLIERS  (target = 1,500 images per class)
  Class                Live count   Multiplier    Projected   vs target
  -------------------- ----------   ----------   ----------   ---------
  No_DR                     1,264            1x        1,264       -236
  Mild                        759            2x        1,518   +      18
  Moderate                    699            2x        1,398       -102
  Severe                      635            2x        1,270       -

In [25]:
  !python src/train.py --data-root data/split --output-dir /kaggle/working/dr_final_model --stage1-lr 0.001 --stage2-lr 0.0001 --unfreeze-blocks 3 --weight-decay 0.0001 --stage1-epochs 10 --stage2-epochs 20 --fast-loader --batch-size 32 --num-workers 2


[Device] Using: cuda
  GPU: Tesla T4
  VRAM: 15.6 GB

[Mode] Single training run ...
[Data] Fast DataLoader active: using native PyTorch transforms (bypassing CPU HoughCircles).

 [TRAINING SET] CLINICAL CLASS-TO-INDEX VERIFICATION
  Index  Class Name         Grade    Status      
  --------------------------------------------------
  0      No_DR              Grade 0  MATCH (OK)  
  1      Mild               Grade 1  MATCH (OK)  
  2      Moderate           Grade 2  MATCH (OK)  
  3      Severe             Grade 3  MATCH (OK)  
  4      Proliferative_DR   Grade 4  MATCH (OK)  
  Sample counts by clinical class in Training Set:
    Class 0 (No_DR             ):  1264 samples ( 18.4%)
    Class 1 (Mild              ):  1518 samples ( 22.1%)
    Class 2 (Moderate          ):  1398 samples ( 20.4%)
    Class 3 (Severe            ):  1270 samples ( 18.5%)
    Class 4 (Proliferative_DR  ):  1412 samples ( 20.6%)


 [VALIDATION SET] CLINICAL CLASS-TO-INDEX VERIFICATION
  Index  Class Name  

In [27]:
  !python src/evaluate.py --checkpoint /kaggle/working/dr_final_model/best_model_20261002_062026.pth --data-root data/split --output-dir /kaggle/working/dr_final_evaluation --fast-loader


[Device] Using: cuda
[Data] Fast DataLoader active: using native PyTorch transforms.

 [TEST SET] CLINICAL CLASS-TO-INDEX VERIFICATION
  Index  Class Name         Grade    Status      
  --------------------------------------------------
  0      No_DR              Grade 0  MATCH (OK)  
  1      Mild               Grade 1  MATCH (OK)  
  2      Moderate           Grade 2  MATCH (OK)  
  3      Severe             Grade 3  MATCH (OK)  
  4      Proliferative_DR   Grade 4  MATCH (OK)  
  Sample counts by clinical class in Test Set:
    Class 0 (No_DR             ):   270 samples ( 49.2%)
    Class 1 (Mild              ):    55 samples ( 10.0%)
    Class 2 (Moderate          ):   150 samples ( 27.3%)
    Class 3 (Severe            ):    29 samples (  5.3%)
    Class 4 (Proliferative_DR  ):    45 samples (  8.2%)

[Data] Test: 549 images across 5 classes
[Data] Classes (test): ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferative_DR']
[Data] Class to index: {'No_DR': 0, 'Mild': 1, 'Modera